# Load ChromaDB

In [1]:
# Mount Google Drive to access and save files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import chromda db
try:
  import chromadb
except:
  !pip install chromadb
  import chromadb

In [3]:
# Set up client and set path to google drive
client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma_db")

# Initialize movie chunk collection and count entries
collection = client.get_collection(name="movie_chunks")
collection.count()

164714

# Query Embedding Function

In [4]:
# Import sentence transformers
try:
  from sentence_transformers import SentenceTransformer
except:
  !pip install sentence_transformers
  from sentence_transformers import SentenceTransformer

In [ ]:
# Load pretrained model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

In [6]:
def get_query_embedding(query, embedding_model):
  """
  This function takes a input query removes leading and
  lagging whitespaces, then generates an embedding for the query.

  The resulting embedding is converted to a Python list format, making it
  compatible with vector databases such as Chroma.

  Parameters
  ----------
  query: str
    The input query string to be embedded

  Returns
  --------
  embedding_model : object
        A preloaded SentenceTransformer model
        (in this case all-mpnet-base-v2).
  """
  return embedding_model.encode(query.strip()).tolist()

# Retrival Function

In [7]:
def retrieve_documents(query, collection, embedding_model, top_k=5):
    """
    This function takes an input query, generates its embedding,
    and uses that embedding to retrieve the top-k most relevant
    document chunks from a Chroma collection.

    It returns both the retrieved document texts and their
    corresponding chunk IDs, which can be used for source
    referencing in a RAG prompt.

    Parameters
    ----------
    query : str
        The input query string used for retrieval

    collection : object
        The Chroma collection containing embedded document chunks

    model : object
        A preloaded SentenceTransformer model used to generate the query embedding
    top_k : int, optional
        The number of top matching document chunks to retrieve, the default is 5

    Returns
    -------
    tuple
        A tuple containing:
          documents : list
              A list of the retrieved documents
          ids : list
              A corresponding list of the chunk IDs for the retrieved documents
    """
    # Call query embedding function
    query_embedding = get_query_embedding(query, embedding_model)

    # Query chroma db
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents"]
    )
    # Return docs and ids
    return results["documents"][0], results["ids"][0]

## Test Cases

In [8]:
# Define function for printing outputs from test cases
def print_test_output(documents, ids):
  """
  Doc-string here....
  """
  # View each result
  for i in range(len(documents)):

      # Shorten plots
      preview = documents[i][:300]
      if len(documents[i]) > 300:
          preview += "..."

      # Print result
      print(f"Result {i+1}")
      print(f"ID: {ids[i]}")
      print(f"{preview}")
      print("-" * 80)

### Title-Based

#### 1. What is the plot of The Hunger Games?

In [9]:
query1 = "What is the plot of The Hunger Games?"
documents, ids = retrieve_documents(query1, collection, embedding_model, top_k=3)

In [10]:
# View each result
print_test_output(documents, ids)

Result 1
ID: 31186339_plot_2
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Katniss has Rue draw them off, then destroys the stockpile by setting off the mines planted around it. Furious, Cato kills the boy assigned to guard it. As Katniss runs from the scene, she hears Rue calling her nam...
--------------------------------------------------------------------------------
Result 2
ID: 31186339_plot_5
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Haymitch warns Katniss that she has made powerful enemies after her display of defiance. She and Peeta return to District 12, while Crane is locked in a room with a bowl of nightlock berries, and President Snow con...
--------------------------------------------------------------------------------
Result 3
ID: 31186339_plot_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: The nation of Panem consists of a wealthy Capitol and twe

#### 2. What happens in the Titanic?

In [ ]:
query2 = "What happens in the Titanic?"
documents, ids = retrieve_documents(query2, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 6524086_plot_0
Movie Title: A Night to Remember
Wiki ID: 6524086
Realse Date: 1958-07-01
Plot Summary: The Titanic was the largest vessel afloat, and was widely believed to be unsinkable. Her passengers included the cream of American and British society. The story of her sinking is told from the point of view of h...
--------------------------------------------------------------------------------
Result 2
ID: 52371_plot_3
Movie Title: Titanic
Wiki ID: 52371
Realse Date: 1997-11-01
Plot Summary: After exhausting his ammunition, Cal realizes to his chagrin that he gave his coat with the diamond to Rose. With the situation now dire, he returns to the boat deck and boards a lifeboat by pretending to look after a lost chi...
--------------------------------------------------------------------------------
Result 3
ID: 6524086_plot_1
Movie Title: A Night to Remember
Wiki ID: 6524086
Realse Date: 1958-07-01
Plot Summary: Fortunately, the radio operator on the receives the distress


#### 3. Give me the story of Star Wars Episode V: The Empire Strikes Back.

In [ ]:
query3 = "Give me the story of Star Wars Episode V: The Empire Strikes Back."
documents, ids = retrieve_documents(query3, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 53964_meta_0
Movie Title: Star Wars Episode V: The Empire Strikes Back
Wiki ID: 53964
Release Date: 1980-05-21
Runtime: 124.0
Languages: English
Countries: United States of America
Genres: Science Fiction, Adventure, Space opera, Fantasy, Family Film, Action
Box Office Revenue: 538375067.0
--------------------------------------------------------------------------------
Result 2
ID: 53964_plot_2
Movie Title: Star Wars Episode V: The Empire Strikes Back
Wiki ID: 53964
Realse Date: 1980-05-21
Plot Summary: Boba Fett secretly follows the Millennium Falcon to the planet Bespin and arrives just before Han and Leia. Bespin is run by Han's old friend Lando Calrissian , but shortly after they arriv...
--------------------------------------------------------------------------------
Result 3
ID: 50744_plot_2
Movie Title: Star Wars Episode VI: Return of the Jedi
Wiki ID: 50744
Realse Date: 1983-05-25
Plot Summary: Palpatine tempts Luke to give in to his anger and join the dark side, a

# Answer Generating

In [ ]:
def generate_answer(query, context, context_ids):
  pass

# Full RAG Pipeline

In [ ]:
def rag_pipeline(query):
  context, context_ids = retrieve_documents(query, collection, embedding_model, top_k=3)
  answer = generate_answer(query, context, context_ids)
  pass